In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import umap
import sentencepiece as spm

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

from keras.utils import plot_model

from transformer_block import TransformerBlock
from tokenization_and_embedding import TokenAndPositionEmbedding

2026-03-24 18:34:56.183696: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 18:34:56.932280: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-24 18:34:59.506207: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
PAD_ID = 3
TOKEN_LENGTH = 128
BATCH_SIZE = 128
SENTENCE_VECTORS = 8000

df = pd.read_parquet("data.parquet")

X = df["text"].astype(str)
y = df["label"].astype(int)

emotion_map = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'} 
df['emotion'] = df['label'].map(emotion_map)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

y_test_np = y_test.to_numpy()


In [4]:
def encode_for_transformer(sp, texts, max_len=128):
    input_ids = []
    masks = []
    pad_id = sp.pad_id()

    for text in texts:
        ids = sp.encode(text.strip(), out_type=int)[:max_len]
        mask = [1] * len(ids)

        while len(ids) < max_len:
            ids.append(pad_id)
            mask.append(0)

        input_ids.append(ids)
        masks.append(mask)

    return (
        np.array(input_ids, dtype=np.int32),
        np.array(masks, dtype=np.int32)
    )

def encode_for_rnn(sp, texts, max_len=128):
    input_ids = []
    masks = []
    pad_id = sp.pad_id()

    for text in texts:
        ids = sp.encode(text.strip(), out_type=int)[:max_len]
        mask = [1] * len(ids)

        while len(ids) < max_len:
            ids.append(pad_id)
            mask.append(0)

        input_ids.append(ids)
        masks.append(mask)

    return (
        np.array(input_ids, dtype=np.int32),
        np.array(masks, dtype=np.int32)
    )

In [5]:
def save_projection_html(X_2d, y, emotion_map, title, out_html):
    df_plot = pd.DataFrame({
        "x": X_2d[:, 0],
        "y": X_2d[:, 1],
        "label": y.astype(int),
        "emotion": [emotion_map[int(i)] for i in y.astype(int)]
    })

    fig = px.scatter(
        df_plot,
        x="x",
        y="y",
        color="emotion",
        hover_data=["label", "emotion"],
        title=title
    )

    fig.update_traces(marker=dict(size=5, opacity=0.6))
    fig.write_html(out_html, include_plotlyjs="cdn")
    print(f"Saved: {out_html}")

In [6]:
def get_feature_model(model):
    return tf.keras.Model(
        inputs=model.input,
        outputs=model.layers[-3].output
    )

In [7]:
def run_projections_for_vectors(vectors, labels, model_name, limit=8000):
    n = min(limit, len(vectors), len(labels))
    X_sub = vectors[:n]
    y_sub = labels[:n]

    print(f"{model_name} vectors shape: {vectors.shape}")
    print(f"Using first {n} samples for t-SNE / UMAP")

    # PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(vectors)
    save_projection_html(
        X_pca,
        labels,
        emotion_map,
        title=f"PCA of {model_name} Sentence Vectors",
        out_html=f"{model_name.lower().replace(' ', '_')}_pca.html"
    )

    # t-SNE
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X_sub)
    save_projection_html(
        X_tsne,
        y_sub,
        emotion_map,
        title=f"t-SNE of {model_name} Sentence Vectors (First {n} Samples)",
        out_html=f"{model_name.lower().replace(' ', '_')}_tsne.html"
    )

    # UMAP
    reducer = umap.UMAP(n_components=2, min_dist=0.1, random_state=42)
    X_umap = reducer.fit_transform(X_sub)
    save_projection_html(
        X_umap,
        y_sub,
        emotion_map,
        title=f"UMAP of {model_name} Sentence Vectors (First {n} Samples)",
        out_html=f"{model_name.lower().replace(' ', '_')}_umap.html"
    )

    # LDA
    lda = LinearDiscriminantAnalysis(n_components=2)
    X_lda = lda.fit_transform(vectors, labels)
    save_projection_html(
        X_lda,
        labels,
        emotion_map,
        title=f"LDA of {model_name} Sentence Vectors",
        out_html=f"{model_name.lower().replace(' ', '_')}_lda.html"
    )

    return {
        "pca": X_pca,
        "tsne": X_tsne,
        "umap": X_umap,
        "lda": X_lda
    }

In [8]:
sp_bpe = spm.SentencePieceProcessor()
sp_bpe.load("m_bpe.model")

sp_uni = spm.SentencePieceProcessor()
sp_uni.load("m_uni.model")

X_test_bpe_ids, X_test_bpe_mask = encode_for_transformer(
    sp_bpe,
    X_test.tolist(),
    max_len=TOKEN_LENGTH
)

X_test_uni_ids, X_test_uni_mask = encode_for_rnn(
    sp_uni,
    X_test.tolist(),
    max_len=TOKEN_LENGTH
)

In [9]:
att_model = tf.keras.models.load_model(
    "best_att.keras",
    custom_objects={
        "TokenAndPositionEmbedding": TokenAndPositionEmbedding,
        "TransformerBlock": TransformerBlock
    },
    compile=False
)

bigru_model = tf.keras.models.load_model(
    "08.03.26_emotion_bigru.keras",
    compile=False
)

# bigru_att_model = tf.keras.models.load_model(
#     "best_bigru_att.keras",
#     compile=False,
#     safe_mode=False
# )

I0000 00:00:1774377345.986330     103 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5582 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
/home/duncanskilton/tf-gpu/lib/python3.10/site-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'transformer_block_4', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/home/duncanskilton/tf-gpu/lib/python3.10/site-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'transformer_block_5', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, des

In [10]:
att_feature_model = get_feature_model(att_model)
bigru_feature_model = get_feature_model(bigru_model)
# bigru_att_feature_model = get_feature_model(bigru_att_model)

In [ ]:
plot_model(att_feature_model, to_file="att_model.png")

In [ ]:
plot_model(bigru_feature_model, to_file="bigru_model.png")

In [13]:
bigru_feature_model = get_feature_model(bigru_model)

print("\n=== BiGRU input inspection ===")
print("bigru_model.inputs:", bigru_model.inputs)
print("bigru_feature_model.inputs:", bigru_feature_model.inputs)
print("number of inputs:", len(bigru_feature_model.inputs))

for i, inp in enumerate(bigru_feature_model.inputs):
    print(f"Input {i}: name={inp.name}, shape={inp.shape}, dtype={inp.dtype}")


=== BiGRU input inspection ===
bigru_model.inputs: [<KerasTensor shape=(None, 128), dtype=int32, sparse=False, ragged=False, name=input_ids>]
bigru_feature_model.inputs: [<KerasTensor shape=(None, 128), dtype=int32, sparse=False, ragged=False, name=input_ids>]
number of inputs: 1
Input 0: name=input_ids, shape=(None, 128), dtype=int32


In [ ]:
print("bigru_model inputs:", bigru_model.inputs)
print("bigru_model input count:", len(bigru_model.inputs))

bigru_feature_model = get_feature_model(bigru_model)

print("bigru_feature_model inputs:", bigru_feature_model.inputs)
print("bigru_feature_model input count:", len(bigru_feature_model.inputs))

bigru_model.summary()

In [ ]:
att_vectors = att_feature_model.predict(
    (X_test_bpe_ids, X_test_bpe_mask),
    batch_size=BATCH_SIZE,
    verbose=1
)

bigru_vectors = bigru_feature_model.predict(
    (X_test_uni_ids, X_test_uni_mask),
    batch_size=BATCH_SIZE,
    verbose=1
)

# bigru_att_vectors = bigru_att_feature_model.predict(
#     X_test_uni_ids,
#     batch_size=BATCH_SIZE,
#     verbose=1
# )

print("Transformer vectors:", att_vectors.shape)
print("BiGRU vectors:", bigru_vectors.shape)
# print("BiGRU + Attention vectors:", bigru_att_vectors.shape)

In [ ]:
att_proj = run_projections_for_vectors(
    att_vectors,
    y_test_np,
    model_name="Transformer"
)

bigru_proj = run_projections_for_vectors(
    bigru_vectors,
    y_test_np,
    model_name="BiGRU"
)

# bigru_att_proj = run_projections_for_vectors(
#     bigru_att_vectors,
#     y_test_np,
#     model_name="BiGRU_Attention"
# )

In [ ]:
N = min(SENTENCE_VECTORS, len(y_test_np), len(att_vectors), len(bigru_vectors), len(bigru_att_vectors))

reducer = umap.UMAP(n_components=2, min_dist=0.1, random_state=42)

X_umap_att = reducer.fit_transform(att_vectors[:N])
X_umap_bigru = reducer.fit_transform(bigru_vectors[:N])
X_umap_bigru_att = reducer.fit_transform(bigru_att_vectors[:N])

df_att = pd.DataFrame({
    "x": X_umap_att[:, 0],
    "y": X_umap_att[:, 1],
    "emotion": [emotion_map[int(i)] for i in y_test_np[:N].astype(int)],
    "model": "Transformer"
})

df_bigru = pd.DataFrame({
    "x": X_umap_bigru[:, 0],
    "y": X_umap_bigru[:, 1],
    "emotion": [emotion_map[int(i)] for i in y_test_np[:N].astype(int)],
    "model": "BiGRU"
})

# df_bigru_att = pd.DataFrame({
#     "x": X_umap_bigru_att[:, 0],
#     "y": X_umap_bigru_att[:, 1],
#     "emotion": [emotion_map[int(i)] for i in y_test_np[:N].astype(int)],
#     "model": "BiGRU + Attention"
# })

df_all = pd.concat([df_att, df_bigru, df_bigru_att], ignore_index=True)

fig = px.scatter(
    df_all,
    x="x",
    y="y",
    color="emotion",
    facet_col="model",
    title=f"UMAP Sentence Vectors — Transformer vs BiGRU vs BiGRU+Attention (N={N})",
    opacity=0.6
)

fig.update_traces(marker=dict(size=5))
fig.write_html(f"{N}_umap_three_model_comparison.html", include_plotlyjs="cdn")
print("Saved: three-model UMAP comparison HTML")